# XAI Evaluation using Quantus and Captum

In this lab, we will explore how to evaluate explanation using Quantus (library specific for XAI eval) and Captum (XAI library that adds explanations eval routines).

**Remark:** pytorch-grad-cam adds a few methods for XAI evaluation but they are not exhaustive and their implementation is not aligned with the main XAI eval frameworks. Refrain from using this library for the eval.

In [ ]:
!pip install -q quantus tensorflow torch torchvision

import os
import glob
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
import quantus
from PIL import Image
from torchvision import transforms

import matplotlib.pyplot as plt


First, we unpack the images and evaluations from the previous lab (ImageNette, Grad-CAM generated on the pretrained ResNet-18).

In [ ]:

# 1. Unzip the explanations file
print("Unzipping explanations.zip...")
!unzip -q -o /content/explanations.zip -d /content/explanations_extracted

# Update paths based on extraction structure
base_dir = "/content/explanations_extracted"
assert os.path.exists(base_dir)

images_dir = os.path.join(base_dir, "original")
explanations_dir = os.path.join(base_dir, "gradcam")

# 2. Load and preprocess the 100 images and explanations
print("Loading images and explanations...")
image_paths = sorted(glob.glob(os.path.join(images_dir, "*")))[:100]
explanation_paths = sorted(glob.glob(os.path.join(explanations_dir, "*")))[:100]


We preprocess images and explanations so that they are ready to be added to the evaluation pipeline

In [ ]:
# Image transformations (ResNet-18 standard)
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

x_list = []
saliency_list = []

# Using the active file lists 'image_paths' and 'explanation_paths' loaded in the environment
for img_fp, exp_fp in zip(image_paths, explanation_paths):
    # Load and preprocess image
    img = Image.open(img_fp).convert("RGB")
    x_list.append(transform(img).numpy())

    # Load and preprocess explanation (saliency map)
    if exp_fp.endswith(".npy"):
        exp = np.load(exp_fp)
    else:
        exp = Image.open(exp_fp).convert("L")
        exp = exp.resize((224, 224))
        exp = np.array(exp) / 255.0
    saliency_list.append(exp)

x_batch = np.array(x_list).astype(np.float32)
saliency_batch = np.array(saliency_list).astype(np.float32)

# Assume dummy targets for model predictions
y_batch = np.zeros(len(x_batch), dtype=int)

print(f"Loaded {len(x_batch)} images with shape {x_batch.shape}")
print(f"Loaded {len(saliency_batch)} explanations with shape {saliency_batch.shape}")


### Faithfulness check

We use pixel flipping for this experiment.

We limit it to the first **10 images** for computational reasons

In [ ]:
n_eval = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define model
model = torchvision.models.resnet18(pretrained=True)
model.to(device)
model.eval()

# Get the model's actual predicted class indices for the evaluation batch
with torch.no_grad():
    inputs_tensor = torch.tensor(x_batch[:n_eval]).to(device)
    outputs = model(inputs_tensor)
    predicted_classes = torch.argmax(outputs, dim=1).cpu().numpy()

print(f"Predicted classes for evaluation: {predicted_classes}")

# 1. Run standard evaluation on batch using predicted classes
print("\n--- Evaluating Faithfulness (Pixel Flipping) on Batch (Observed predicted classes) ---")
pixel_flipping = quantus.PixelFlipping(perturb_baseline="black", features_in_step=16)

faithfulness_scores = pixel_flipping(model=model,
                                     x_batch=x_batch[:n_eval],
                                     y_batch=predicted_classes,
                                     a_batch=saliency_batch[:n_eval],
                                     device=device)

print(f"Pixel Flipping Mean Score: {np.mean(faithfulness_scores):.4f}")

In [ ]:
# 2. Plot Pixel Flipping curve for a single image (observing the predicted class)
print("\n--- Plotting Pixel Flipping for a Single Image ---")
single_x = x_batch[0:1]
single_a = saliency_batch[0:1]

# Get the model's actual predicted class index for this image
with torch.no_grad():
    inputs_tensor = torch.tensor(single_x).to(device)
    outputs = model(inputs_tensor)
    predicted_class_idx = torch.argmax(outputs, dim=1).cpu().numpy()

print(f"The actual predicted class index being evaluated is: {predicted_class_idx[0]}")

# Run pixel flipping on the single image observing the predicted class index
single_score = pixel_flipping(model=model,
                              x_batch=single_x,
                              y_batch=predicted_class_idx,
                              a_batch=single_a,
                              device=device)




In [ ]:
steps = np.arange(0, len(single_score[0]))
plt.figure(figsize=(8, 5))
plt.plot(steps, single_score[0], marker='o', markersize=3, color='purple', linestyle='-', linewidth=1)
plt.title(f"Pixel Flipping Faithfulness Curve (Observed Class: {predicted_class_idx[0]})")
plt.xlabel("Perturbation Step (Flipped Patches)")
plt.ylabel("Model Output Probability / Score")
plt.grid(True)
plt.show()

### Localization check

We execute localization on phony bounding box coordinates (since ImageNette does not provide segmentation maps or bounding boxes).

You should change these maps to your own ones.

In [ ]:
# 5. Evaluate Localization using Pointing Game

# Create binary segmentation masks (NumPy arrays) representing the fallback targets/bounding boxes
s_batch_list = []
for _ in range(len(x_batch[:100])):
    # Initialize a blank binary mask of size (224, 224)
    mask = np.zeros((224, 224), dtype=np.float32)
    # Set a fallback central bounding box (from pixel 50 to 170) to 1.0
    mask[50:170, 50:170] = 1.0
    s_batch_list.append(mask)

# We convert to a NumPy array and add a channel dimension so that it matches x_batch's dimensions (100, 1, 224, 224)
s_batch = np.array(s_batch_list)
s_batch = np.expand_dims(s_batch, axis=1)

# Get model predictions for targets to evaluate on predicted classes
with torch.no_grad():
    inputs_tensor = torch.tensor(x_batch[:100]).to(device)
    outputs = model(inputs_tensor)
    predicted_classes_100 = torch.argmax(outputs, dim=1).cpu().numpy()

print("\n--- Evaluating Localization (Pointing Game) ---")
pointing_game = quantus.PointingGame()

# Pass the binary mask NumPy array as s_batch
localization_scores = pointing_game(
    model=model,
    x_batch=x_batch[:100],
    y_batch=predicted_classes_100,
    a_batch=saliency_batch[:100],
    s_batch=s_batch,
    device="cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Pointing Game Mean Score: {np.mean(localization_scores):.4f}")

## Evaluating multiple metrics and explanations at the same time

Check out the [Quantus docs](https://quantus.readthedocs.io/en/latest/getting_started/getting_started_example.html#large-scale-evaluations) for understanding how to make the large-scale evaluation of metrics and explanations more efficient.

### Captum for evaluating the explanations

Captum allows for generation + evaluation using the same library.
We will regenerate and evaluate the explanations.

The issue with Captum is that it only supports 2 metrics:

1. Infidelity (metric for faithfulness, see [paper](https://arxiv.org/abs/1901.09392)), which we showcase here
2. Sensitivity (metric for robustness, see [paper](https://openreview.net/forum?id=H1xJjlbAZ))

In [ ]:
# 6. Evaluate Explanations using Captum
!pip install -q captum

import torch
import numpy as np
from captum.metrics import infidelity, sensitivity_max
from captum.attr import LayerGradCam

# Ensure model is in eval mode and moved to the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# Prepare input tensors (first 10 images)
inputs = torch.tensor(x_batch[:10]).to(device)

# Get the model's actual predicted class indices for these inputs
with torch.no_grad():
    outputs = model(inputs)
    predicted_targets = torch.argmax(outputs, dim=1)

print(f"Predicted classes for evaluation: {predicted_targets.cpu().numpy()}")

# Initialize LayerGradCam targeting ResNet-18's final convolutional layer (layer4)
gradcam = LayerGradCam(model, model.layer4)

# Compute Grad-CAM attributions
# We interpolate/attribute to the input shape to match the input dimensions for metrics
attributions = gradcam.attribute(inputs, target=predicted_targets, attribute_to_layer_input=False)
attributions = LayerGradCam.interpolate(attributions, (224, 224))

# Expand attribution channel dimension from 1 to 3 to match the inputs tensor shape [10, 3, 224, 224]
attributions_expanded = attributions.expand(-1, 3, -1, -1)

# Define a simple perturbation function for Infidelity
def perturb_fn(inputs):
    # Add small Gaussian noise as a perturbation
    noise = torch.randn_like(inputs) * 0.01
    return inputs - noise, noise

# Compute Infidelity metric
print("\n--- Evaluating Infidelity using Captum ---")
infidelity_scores = infidelity(
    forward_func=model,
    perturb_func=perturb_fn,
    inputs=inputs,
    attributions=attributions_expanded,
    target=predicted_targets
)
print(f"Captum Infidelity Mean Score: {infidelity_scores.mean().item():.4f}")
